In [7]:
# 📦 라이브러리 임포트
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import (Conv2D, BatchNormalization, Activation, 
                          MaxPooling2D, Dropout, GlobalAveragePooling2D, 
                          Dense, Input)
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.regularizers import l2
from sklearn.model_selection import train_test_split

# 📥 데이터 로딩 및 분할
(x, y), (_, _) = cifar10.load_data()
x_data, tt_x, y_data, tt_y = train_test_split(x, y, train_size=0.85, stratify=y)
tr_x, val_x, tr_y, val_y = train_test_split(x_data, y_data, train_size=0.82, stratify=y_data)

# 🎨 정규화
s_tr_x = tr_x / 255.0
s_val_x = val_x / 255.0
s_tt_x = tt_x / 255.0

# 🧠 모델 구조 정의
def conv_block(model, filters, dropout_rate):
    model.add(Conv2D(filters, 3, padding='same', kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(filters, 3, padding='same', kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(2))
    model.add(Dropout(dropout_rate))
    return model

model = Sequential()
model.add(Input(shape=(32, 32, 3)))
model = conv_block(model, 64, 0.3)
model = conv_block(model, 128, 0.4)
model = conv_block(model, 256, 0.5)
model.add(GlobalAveragePooling2D())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# ⚙️ 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# 🧪 데이터 증강
datagen = ImageDataGenerator(
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    zoom_range=0.1
)

# 📍 콜백 설정
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# 🚀 학습
history = model.fit(
    datagen.flow(s_tr_x, tr_y, batch_size=64),
    validation_data=(s_val_x, val_y),
    epochs=100,
    callbacks=callbacks
)

# 🧾 테스트 평가
test_loss, test_acc = model.evaluate(s_tt_x, tt_y)
print(f"✅ 테스트 정확도: {test_acc:.4f}, 손실: {test_loss:.4f}")

Epoch 1/100


/opt/anaconda3/envs/stenv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.2938 - loss: 2.0291
Epoch 1: val_accuracy improved from -inf to 0.34497, saving model to best_model.keras
545/545 ━━━━━━━━━━━━━━━━━━━━ 72s 131ms/step - accuracy: 0.2940 - loss: 2.0287 - val_accuracy: 0.3450 - val_loss: 1.9503 - learning_rate: 0.0010
Epoch 2/100
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.5003 - loss: 1.4887
Epoch 2: val_accuracy improved from 0.34497 to 0.40627, saving model to best_model.keras
545/545 ━━━━━━━━━━━━━━━━━━━━ 72s 131ms/step - accuracy: 0.5004 - loss: 1.4886 - val_accuracy: 0.4063 - val_loss: 2.1424 - learning_rate: 0.0010
Epoch 3/100
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.5708 - loss: 1.3028
Epoch 3: val_accuracy improved from 0.40627 to 0.57111, saving model to best_model.keras
545/545 ━━━━━━━━━━━━━━━━━━━━ 72s 132ms/step - accuracy: 0.5709 - loss: 1.3028 - val_accuracy: 0.5711 - val_loss: 1.2751 - learning_rate: 0.0010
Epoch 4/100
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms